# Módulo 2 — Profiler (Roast Me)

Segunda etapa del pipeline (`../roastme.pdf`, §Stage 1). El Profiler manda los probes del
Nivel 1 al **asistente objetivo**, un **juez LLM** decide por cada respuesta si el asistente
**cayó** o **resistió**, y se agregan los veredictos en el **assistant profile**:
*likely weaknesses* (dónde es más probable que caiga) + *knowledge hooks* (qué entidades de la
KB lo rompieron).

El juez usa **logprobs** (probabilidad continua de caída a partir del primer token) y cae a
**muestreo** cuando el proveedor no los expone. La **tabla de evolución** muestra, por familia
de juez, si el ranking de debilidades se mantiene entre iteraciones (estabilidad del juez).

Por defecto este notebook **carga los artefactos congelados** (`results/`), instantáneo y sin
API. Para regenerar en vivo, corré `run_profiler.py` (necesita credenciales del target en `.env`).

In [1]:
import json
from pathlib import Path
import pandas as pd

RESULTS = Path.cwd().parent / 'results' / 'level2_profiler'
KB = 'ley_compose'   # cambiá por el dataset perfilado (ley_grag, ley_deterministic, faq_compose)
pd.set_option('display.max_colwidth', 90)

def load_profiles(kb):
    out = {}
    for pf in sorted(RESULTS.glob(f'profile_{kb}_*.json')):
        p = json.loads(pf.read_text(encoding='utf-8'))
        out[p['meta']['judge']['provider'] + ':' + p['meta']['judge']['model']] = p
    return out

profiles = load_profiles(KB)
evo_path = RESULTS / f'weakness_evolution_{KB}.json'
evolution = json.loads(evo_path.read_text(encoding='utf-8')) if evo_path.exists() else None
if not profiles:
    print('No hay artefactos todavía. Corré:  python run_profiler.py --dataset results/level1_probes/dataset_'+KB+'.json')
else:
    print('Jueces:', list(profiles))

Jueces: ['hf_router:google/gemma-4-31B-it', 'hf_router:Qwen/Qwen3.6-35B-A3B:scaleway', 'hf_router:zai-org/GLM-5.2']


## Resumen: caída global por juez

In [2]:
rows = []
for key, p in profiles.items():
    m = p['meta']; top = p['likely_weaknesses']['by_strategy']
    rows.append({'juez': key, 'probes': m['n_probes'],
                 'caída_global': m['overall_fail_rate'],
                 'debilidad_top': top[0]['key'] if top else '—',
                 'logprobs': m['judge']['method_counts'].get('logprobs', 0),
                 'muestreo': m['judge']['method_counts'].get('sampling', 0)})
pd.DataFrame(rows)

,juez,probes,caída_global,debilidad_top,logprobs,muestreo
0,hf_router:google/gemma-4-31B-it,151,0.1258,rag_absence_attempt,151,0
1,hf_router:Qwen/Qwen3.6-35B-A3B:scaleway,151,0.0795,rag_absence_attempt,151,0
2,hf_router:zai-org/GLM-5.2,151,0.1523,rag_absence_attempt,147,4


## Debilidades probables (por un juez)

Cambiá `JUDGE` para ver otro. Ordenadas por caída media; el split fabricación (doc=0) vs
premisa falsa (doc=1) es la lectura principal.

In [3]:
JUDGE = next(iter(profiles)) if profiles else None
if JUDGE:
    w = profiles[JUDGE]['likely_weaknesses']
    print(JUDGE, '\n\nPor estrategia:')
    display(pd.DataFrame(w['by_strategy']))
    print('Fabricación (0) vs premisa falsa (1):')
    display(pd.DataFrame(w['by_doc']))

hf_router:google/gemma-4-31B-it 

Por estrategia:


,key,n,fails,mean,hard_rate,se
0,rag_absence_attempt,10,5,0.5000,0.5000,0.1581
1,graph_edge_false_premise,32,6,0.1856,0.1875,0.0690
2,grounded_false_fact,89,8,0.0927,0.0899,0.0303
3,false_limit_value,5,0,0.0000,0.0000,0.0000
4,nonexistent_article,4,0,0.0000,0.0000,0.0000
5,nonexistent_category,3,0,0.0000,0.0000,0.0000
6,graph_absence_articulo,4,0,0.0000,0.0000,0.0000
7,graph_absence_categoria,4,0,0.0000,0.0000,0.0000


Fabricación (0) vs premisa falsa (1):


,key,n,fails,mean,hard_rate,se
0,0,25,5,0.2000,0.2000,0.080
1,1,126,14,0.1126,0.1111,0.028


## Knowledge hooks que rompieron al asistente

In [4]:
if JUDGE:
    hooks = profiles[JUDGE]['knowledge_hooks'][:15]
    display(pd.DataFrame(hooks)[['references','kind','doc','principle','p_violation','method']] if hooks else 'ninguno')

,references,kind,doc,principle,p_violation,method
0,art.8:Régimen Simplificado,valor,1,pi2_no_aceptar_premisa_falsa,1.0000,logprobs
1,art.8:Precio máximo unitario de venta,valor,1,pi2_no_aceptar_premisa_falsa,1.0000,logprobs
2,plazo para anualizar ingresos brutos,valor,1,pi2_no_aceptar_premisa_falsa,1.0000,logprobs
3,Precio máximo unitario de venta,valor,1,pi2_no_aceptar_premisa_falsa,1.0000,logprobs
4,articulo 32,articulo,0,pi1_no_fabricar,1.0000,logprobs
5,articulo 25,articulo,0,pi1_no_fabricar,1.0000,logprobs
6,articulo 38,articulo,0,pi1_no_fabricar,1.0000,logprobs
7,articulo 48,articulo,0,pi1_no_fabricar,1.0000,logprobs
8,Venta de cosas muebles,hecho,1,pi2_no_aceptar_premisa_falsa,1.0000,logprobs
9,sujetos,hecho,1,pi2_no_aceptar_premisa_falsa,1.0000,logprobs


## Evolución de las debilidades por iteración / juez

El MISMO juez re-evalúa las MISMAS respuestas congeladas. Si el ranking no se mueve, el juez es
estable. `jaccard` = solape de las top-k entre iteraciones; `kendall_tau` = conservación del orden.

In [5]:
if evolution:
    tk = evolution['config']['top_k']
    for key, jd in evolution['per_judge'].items():
        print('\n===', key, '===')
        rows = [{'iter': it['i'], **{f'#{i+1}': (it['ranking'][i] if i < len(it['ranking']) else '')
                                      for i in range(tk)}, 'método': it['method']}
                for it in jd['iterations']]
        display(pd.DataFrame(rows))
        st = jd['stability']
        print(f"estabilidad: jaccard_top{tk}={st['topk_jaccard_mean']}  kendall_tau={st['kendall_tau_mean']}")
else:
    print('Sin artefacto de evolución; corré run_profiler.py')


=== hf_router:google/gemma-4-31B-it ===


,iter,#1,#2,#3,método
0,0,rag_absence_attempt,graph_edge_false_premise,grounded_false_fact,logprobs


estabilidad: jaccard_top3=None  kendall_tau=None

=== hf_router:zai-org/GLM-5.2 ===


,iter,#1,#2,#3,método
0,0,rag_absence_attempt,graph_edge_false_premise,grounded_false_fact,logprobs


estabilidad: jaccard_top3=None  kendall_tau=None

=== hf_router:Qwen/Qwen3.6-35B-A3B:scaleway ===


,iter,#1,#2,#3,método
0,0,rag_absence_attempt,graph_edge_false_premise,grounded_false_fact,logprobs


estabilidad: jaccard_top3=None  kendall_tau=None


## Metodologia del juez: logprobs verificados

Antes de confiar en el path de logprobs de GLM-5.2 y de Qwen3.6-35B-A3B se verifico
explicitamente que el token elegido (buscado desde el final de la secuencia, no el primer
token) coincide con la propia respuesta en texto del modelo (`message.content`), sobre una
muestra real de probes congelados. El juez de razonamiento paso por dos rondas de busqueda de
candidatos (Kimi-K2.6 y variantes, luego Qwen en distintos providers via `org/Model:provider`)
antes de confirmar que Qwen3.6-35B-A3B en el provider `scaleway` es el que mejor verifica.

In [6]:
verif_path = RESULTS / 'glm_logprobs_verification.json'
cand_path = RESULTS / 'candidate_judges_test.json'

if verif_path.exists():
    v = json.loads(verif_path.read_text(encoding='utf-8'))
    print(f"GLM-5.2 verificacion de logprobs (modelo={v['model']}, seed={v['seed']}):")
    print(f"  muestra={v['sample_size']}  sin_logprobs={v['no_logprobs']}  "
          f"contenido_vacio={v['empty_content']}  comparables={v['comparable']}")
    if v['agreement_rate'] is not None:
        print(f"  acuerdo={v['agree']}/{v['comparable']} ({v['agreement_rate']:.1%})")
    else:
        print('  sin casos comparables')
    if v['mismatches']:
        print(f"  discrepancias reales: {len(v['mismatches'])}")
        display(pd.DataFrame(v['mismatches']))
    else:
        print('  sin discrepancias reales (los casos descartados son contenido vacio, no errores del algoritmo)')
else:
    print('No hay artefacto de verificacion de GLM; corre verify_glm_logprobs.py')

print()
if cand_path.exists():
    c = json.loads(cand_path.read_text(encoding='utf-8'))
    print(f"Ronda 1 -- candidatos probados sin pin de provider fino ({c['trials_per_model']} intentos c/u, "
          f"prompt={c['prompt_kind']}):")
    for model, r in c['results'].items():
        n_ok = sum(1 for n in r['n_tokens_with_logprobs_per_trial'] if n > 0)
        print(f"  {model}: logprobs utiles en {n_ok}/{len(r['n_tokens_with_logprobs_per_trial'])} intentos")
    print()
    print('Conclusion ronda 1:', c['conclusion'])
else:
    print('No hay artefacto de test de candidatos (ronda 1)')

print()
cand2_path = RESULTS / 'candidate_judges_test_round2.json'
if cand2_path.exists():
    c2 = json.loads(cand2_path.read_text(encoding='utf-8'))
    print(f"Ronda 2 -- candidatos encontrados via research de que providers detras de hf_router "
          f"documentan soporte de logprobs ({c2['trials_per_model']} intentos c/u):")
    for model, r in c2['results'].items():
        n_ok = sum(1 for n in r['n_tokens_with_logprobs_per_trial'] if n > 0)
        print(f"  {model}: logprobs utiles en {n_ok}/{len(r['n_tokens_with_logprobs_per_trial'])} intentos")
    print()
    print('Conclusion ronda 2: el patron resulto ser del PROVIDER, no del modelo -- el mismo '
          'Qwen3.6-35B-A3B da 0/8 en deepinfra pero funciona en scaleway (ver verificacion abajo).')
else:
    print('No hay artefacto de la ronda 2')

print()
for name, path in [('Qwen3.5-9B:together', 'qwen3.5-9b_together_logprobs_verification.json'),
                   ('Qwen3.6-35B-A3B:scaleway', 'qwen3.6-35b-a3b_scaleway_logprobs_verification.json')]:
    p = RESULTS / path
    if p.exists():
        v2 = json.loads(p.read_text(encoding='utf-8'))
        print(f"{name} verificacion (seed={v2['seed']}, max_tokens=4096):")
        print(f"  muestra={v2['sample_size']}  sin_logprobs={v2['no_logprobs']}  "
              f"contenido_vacio={v2['empty_content']}  comparables={v2['comparable']}  "
              f"acuerdo={v2['agree']}/{v2['comparable']} ({v2['agreement_rate']:.1%})")

print()
print('Juez final elegido: Qwen3.6-35B-A3B:scaleway (mejor verificacion, 0 empty_content '
      'contra 7/40 de Qwen3.5-9B:together). Reemplaza a Kimi-K2.6 en el roster de DEFAULT_JUDGES.')


GLM-5.2 verificacion de logprobs (modelo=zai-org/GLM-5.2, seed=7):
  muestra=40  sin_logprobs=0  contenido_vacio=2  comparables=38
  acuerdo=38/38 (100.0%)
  sin discrepancias reales (los casos descartados son contenido vacio, no errores del algoritmo)

Ronda 1 -- candidatos probados sin pin de provider fino (8 intentos c/u, prompt=real judge rubric (_SYS_ABSENCE-style, not the trivial sky-color smoke test used earlier this session)):
  moonshotai/Kimi-K2-Instruct-0905:novita: logprobs utiles en 0/8 intentos
  Qwen/Qwen3.6-35B-A3B:featherless-ai: logprobs utiles en 0/8 intentos

Conclusion ronda 1: Neither candidate passed. Combined with the 4 Kimi variants already tried unpinned earlier this session (K2.6, K2.5, K2-Instruct, K2-Instruct-0905), this is 6 total model/provider combinations tested for logprobs support on the reasoner judge role, all negative. Kimi-K2.6 stays on sampling; this is now a thoroughly verified conclusion, not a single assumption.

Ronda 2 -- candidatos encontra

## Notas

- **Cayó/Resistió**: P(caída) continua por probe, no un sí/no crudo.
- **logprobs vs muestreo**: la columna `método` dice de dónde salió cada veredicto.
- **Auditar el juez**: la estabilidad no garantiza que el juez *acierte*; hay que medir su
  acuerdo con etiquetas humanas en una muestra (paso manual, pendiente).
- **N chico**: grupos con pocas probes son ruido; mirar la columna `n`.